In [12]:
import matplotlib.pyplot as plt
import numpy as np
import random


In [13]:
L = 10                                      # [Mpc] box edge size
Ngal = 1e6                                  # number of galaxies in the box

poisson_catalog = np.random.uniform(low=0, high=L, size=(int(Ngal), 3))  # (Ngal, 3), positions

In [ ]:
Ndiscrete = 100                                   # discretisation number in each direction
eps = 0.5                                         # PV parameter
A = 1                                             # PV signal amplitude 

n = np.arange(Ndiscrete)                          
kf = 2*np.pi/L                                    # fundamental mode value 
k1d = 2*np.pi/L * n                               # modes along one direction      
kx, ky, kz = np.meshgrid(k1d, k1d, k1d, indexing='ij')      # each shape (N, N, N)
k_vecs = np.stack([kx, ky, kz], axis=-1)          # shape (N, N, N, 3)
k_mag = np.linalg.norm(k_vecs, axis=-1)           # (N,N,N)
k_mag[0, 0, 0] = 1.0                              # avoid division by zero for the k=0 mode
khat = k_vecs / k_mag[..., None]                  # (N,N,N,3) normalised k vectors

z = np.array([0.0, 0.0, 1.0])                     # arbitrary reference direction for polarisation basis

e1 = np.cross(z, khat)                            # (N,N,N,3)
bad = np.linalg.norm(e1, axis=-1) < 1e-12         # where khat // z
e1[bad] = np.cross([1.0, 0.0, 0.0], khat[bad])    # use x instead of z for those cases
norm_e1 = np.linalg.norm(e1, axis=-1, keepdims=True)
norm_e1[norm_e1 < 1e-12] = 1.0                    # avoid division by zero for the k=0 mode
e1 /= norm_e1
e2 = np.cross(khat, e1)                           # (N,N,N,3)

e_plus  = (e1 + 1j*e2) / np.sqrt(2)               # (N,N,N,3) complex
e_minus = (e1 - 1j*e2) / np.sqrt(2)

def power_spectrum(k, kmin=1*kf, kmax=8*kf):    # define the power spectrum according to a model 
    return ((k >= kmin) & (k <= kmax)).astype(float)                        

Pk = power_spectrum(k_mag)                        # (N,N,N)

def gauss():                                      # generate a complex Gaussian random field with zero mean and unit variance
    return np.random.randn(*k_mag.shape) + 1j*np.random.randn(*k_mag.shape)         # (N,N,N)

a_plus  = np.sqrt(Pk*(1+eps)/2) * gauss()         # (N,N,N)
a_minus = np.sqrt(Pk*(1-eps)/2) * gauss()         # (N,N,N)

psi_k = a_plus[..., None]*e_plus + a_minus[..., None]*e_minus   # (N,N,N,3)
psi_k[0, 0, 0] = 0.0                              # global mode set to zero

psi_x = np.fft.ifftn(psi_k[..., 0]).real          # (N,N,N) 
psi_y = np.fft.ifftn(psi_k[..., 1]).real          # (N,N,N) 
psi_z = np.fft.ifftn(psi_k[..., 2]).real          # (N,N,N) 
psi = np.stack([psi_x, psi_y, psi_z], axis=-1)    # (N,N,N,3) real space displacement field

# For each position in the Poisson catalog, find the corresponding displacement vector from the grid
idx = np.floor(poisson_catalog / L * Ndiscrete).astype(int) % Ndiscrete   # (Ngal, 3)
ix, iy, iz = idx[:, 0], idx[:, 1], idx[:, 2]                              # each (Ngal,)
disp = psi[ix, iy, iz]                                                    # (Ngal, 3) 

poisson_catalog_displaced = (poisson_catalog + A * disp) % L              # (Ngal, 3)

In [17]:
print(np.mean(np.linalg.norm(disp, axis=-1)))

0.0009077682530833833
